# Megaline Mobile Project by Dwayne Pringle

#### A basic machince learning project
This project focuses on analyzing subscriber behavior for Megaline, a mobile carrier, with the objective of recommending optimal mobile plans. By leveraging historical usage data, we aim to build a classification model that predicts whether a subscriber is better suited for the SMART or ULTRA plan. The goal is to improve customer satisfaction and optimize plan assignments through data-driven insights.

#### Work Plan
1. **Import Libraries**
    - Import Python libraries needed for data analysis, visualization, and machine learning model building
2. **Explore the Data**
    - Load each CSV dataset
    - Review dataset structure, columns, and data types
4. **Clean the Data**
    - Identify and remove duplicate records
    - Identify and handle missing values
5. **Model Development**
    - Split the source data into training, validation, and test datasets
6. **Model Training**
    - Train Random Forest model
    - Train Logistic Regression model
7. **Model Evaluation**
    - Evaluate and compare model performance
    - Check the quality and accuracy of the models
8. **Conclusion**
    - Summarize findings, model results, and business insights

## Importing Files and Analyzing Data

In [33]:
# Imports
import pandas as pd
import numpy as np 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

In [34]:
# Importing File
df = pd.read_csv('./dataset/users_behavior.csv')

In [35]:
# Analyzing Data File
df.info()
df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0
5,58.0,344.56,21.0,15823.37,0
6,57.0,431.64,20.0,3738.90,1
7,15.0,132.40,6.0,21911.60,0
8,7.0,43.39,3.0,2538.67,1
9,90.0,665.41,38.0,17358.61,0


In [36]:
# Check for missing values
df.isnull().sum()

calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

In [37]:
# Checking for duplicates
df.duplicated().sum()

np.int64(0)

## Spliting the Source Data

Here we are spliting the source data into a training set, a validation set, and a test set.

In [38]:
# Split the source data into a training set, a validation set, and a test set.
features = df.drop(['is_ultra'],axis= 1)
target = df['is_ultra']

# Spliting the Training Set into two 
features_train, features_tmp, target_train, target_tmp = train_test_split (features, target, test_size= 0.20, random_state =12345)

In [39]:
# Splitting the data into Validation and Test sets
features_valid, features_test, target_valid, target_test = train_test_split(features_tmp, target_tmp, test_size = 0.5, random_state= 12345)

## Let's Investigate

Investigate the quality of different models by changing hyperparameters. 

### RandomForest

In [40]:
# Random Forest to change the hyperparaters
for n in [10, 50, 100]:
    for depth in [3, 5, 10]:
        model = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42)
        model.fit(features_train, target_train)
        val_pred = model.predict(features_valid)
        acc = accuracy_score(target_valid, val_pred)
        print(f"n_estimators={n}, max_depth={depth} => Validation Accuracy: {acc:.4f}")

n_estimators=10, max_depth=3 => Validation Accuracy: 0.7913
n_estimators=10, max_depth=5 => Validation Accuracy: 0.7913
n_estimators=10, max_depth=10 => Validation Accuracy: 0.7913
n_estimators=50, max_depth=3 => Validation Accuracy: 0.7850
n_estimators=50, max_depth=5 => Validation Accuracy: 0.7975
n_estimators=50, max_depth=10 => Validation Accuracy: 0.7975
n_estimators=100, max_depth=3 => Validation Accuracy: 0.7882
n_estimators=100, max_depth=5 => Validation Accuracy: 0.8006
n_estimators=100, max_depth=10 => Validation Accuracy: 0.8006


### Logistic Regression

In [41]:
# Using Logistic Regression to change the hyperparaters
model = LogisticRegression(random_state=54321, solver='liblinear') # initialize logistic regression constructor with parameters random_state=54321 and solver='liblinear'
model.fit(features_train, target_train)  # train model on training set

# Evaluate the model on the training and validation sets
score_train = model.score(features_train, target_train)  
score_valid = model.score(features_valid, target_valid)  

# Print the accuracy scores for both sets
print("Accuracy of the logistic regression model on the training set:",score_train)
print("Accuracy of the logistic regression model on the validation set:",score_valid)

Accuracy of the logistic regression model on the training set: 0.7016725009723843
Accuracy of the logistic regression model on the validation set: 0.67601246105919


### Brief Description of My Findings
Evaluating the model performance for Megaline’s plan classification task, I conducted hyperparameter tuning on a Random Forest classifier by varying n_estimators and max_depth. For each configuration, the model was trained on the training data and evaluated on the validation set to measure accuracy. Separately, a Logistic Regression model was trained using a fixed configuration with random_state=54321 and solver='liblinear', and its performance was also assessed on the training and validation sets.

After comparing validation accuracies from both approaches, the Random Forest model consistently outperformed Logistic Regression across the tested configurations. The Random Forest model achieved the highest validation accuracy, indicating better generalization and predictive quality. Therefore, Random Forest was selected as the final model for Megaline’s classification needs, as it demonstrated superior performance during the model selection process.

## Checking the quality of the model

Here I'm checking the quality of the model using the test set.

In [42]:
# Train the final model on the training set and evaluate it on the test set
final_model = RandomForestClassifier(random_state=54321,max_depth=10 , n_estimators=100) # change n_estimators to get best model

# Train the final model on the training set
final_model.fit(features_train, target_train)

# Predict on test set
predictions = final_model.predict(features_test)

# Evaluate the model on the test set
print("Accuracy:", accuracy_score(target_test, predictions))

Accuracy: 0.7981366459627329


## Conclusion:

In this project, we analyzed subscriber behavior for Megaline to recommend the most suitable mobile plan—either SMART or ULTRA—using a classification model trained on historical usage data. After evaluating multiple models, the Random Forest classifier with max_depth=10 and n_estimators=100 emerged as the best performer based on validation accuracy.

The final model was trained on the training data and evaluated on the test set, yielding an accuracy of 0.7981. This indicates that the model correctly predicts the appropriate plan for approximately 80% of new customers. These results confirm that the selected model is both reliable and effective for supporting Megaline in making data-driven, customer-focused plan recommendations.